In [ ]:
import sys
print(sys.executable)

In [ ]:
# =============================
# Reqs/setup
# =============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import json
from datetime import datetime

from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

In [ ]:
# NHL API URL
BASE_URL = "https://api-web.nhle.com/v1"

# Test connection
response = requests.get(f"{BASE_URL}/standings/now")
print(f"Status Code: {response.status_code}")
print("API is working!" if response.status_code == 200 else "API error")

In [ ]:
# =============================
# Data Paths
# =============================

DATA_DIR = Path.cwd() / "data"

events_path = DATA_DIR / "events.parquet"
games_path = DATA_DIR / "games.parquet"
players_path = DATA_DIR / "players.parquet"
stints_path = DATA_DIR / "stints.parquet"
tracking_path = DATA_DIR / "tracking.parquet"


In [ ]:
# =============================
# Load Data
# =============================

events = pd.read_parquet(events_path)
games = pd.read_parquet(games_path)
players = pd.read_parquet(players_path)
stints = pd.read_parquet(stints_path)
tracking = pd.read_parquet(tracking_path)

print("Events:", events.shape)
print("Games:", games.shape)
print("Players:", players.shape)
print("Stints:", stints.shape)
print("Tracking:", tracking.shape)

In [ ]:
players.head(10)
#games['home_team'].nunique() 

In [ ]:
# =============================
# Exploration (minimal)
# =============================

# Games
games.head()
teams_tot = games['home_team'].nunique() 
leagues = games["league"].unique()
print("\nGames details:")
print(f"There are {teams_tot} teams in the leagues(s): {leagues}.")
games[["home_score", "away_score"]].max().max()  # max 8 goal win i
games['home_team'].nunique()  #  32 teams in league

# Events
unique_events = events["event_type"].nunique()  # types of events
outcomes_unique = events["outcome"].nunique()  # event outcomes
outcomes_types = events["outcome"].unique()  # types of outcomes
print("Events details:")
tracking_data = events["has_tracking_data"].value_counts(normalize=True)  # Most events (>90%) have tracking data, don't impute missing
print(f"There are {unique_events} types of events with {outcomes_unique}: {outcomes_types}, with tracking data avilable for outcomes as follows: {tracking_data}.")
event_freq = events["event_type"].value_counts(normalize=True)
print(f"Event frequencies are as follows: {event_freq}")
pass_detail_prop = (
    events.loc[events["event_type"] == "pass", "detail"]
          .value_counts(normalize=True)
)
print(f"Pass type frequencies are as follows: {pass_detail_prop}")

# Players
unique_players = players["player_id"].nunique()
print("\nPlayers details:")
print(f"There are {unique_players} players in the league.")

# Stints (shifts). Game stints triggered when player enters or leaves the ice
print("\nStints/shifts details:")
print("Nothing remarkeable to note. Useful to investigate penalties (info on number of skaters on ice available).")

plt.hist(stints["period_time_start"], bins=30, alpha=0.3, label="Stint Start")
plt.hist(stints["period_time_end"], bins=30, alpha=0.3, label="Stint End")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Frequency")
plt.show()

# Tracking (shifts)
tracking.head()
tracking.columns
missing_x_vel_tracking = tracking["tracking_vel_x"].isna().mean()
missing_x_tracking = tracking["tracking_x"].isna().mean()
missing_y_vel_tracking = tracking["tracking_vel_y"].isna().mean()
missing_y_tracking = tracking["tracking_y"].isna().mean()
missing_eventID_tracking = tracking["sl_event_id"].isna().mean()

print("\nTracking details:")
print(f"The number of rows missing values for variables is as follows: x-tracking: {missing_x_tracking}; x-tracking velocity: {missing_x_vel_tracking}: y-tracking: {missing_y_tracking}; y-tracking velocity: {missing_y_vel_tracking}; SL event ID: {missing_eventID_tracking}")


In [ ]:
## Subset `events` object to defensive passes

# Defensive zone threshold (adjust if needed
DZ_THRESHOLD = -25

dz_passes = (
    events
    .query("event_type == 'pass'") # Subset to passes
    .query("detail in ['d2d', 'd2doffboards', 'outlet', 'outletoffboards', 'stretch', 'stretchoffboards', 'ozentrystretch', 'ozentrystretchoffboards']") # Subset to defensive passes
    .query("x_adj < @DZ_THRESHOLD") # subset rows to defensive zone
    .copy()
)

dz_passes.shape


In [ ]:
# Investigate success rates of different defensive pass types 

pass_overall_breakdown = dz_passes["detail"].value_counts(normalize=True)
dz_passes["outcome"].value_counts(normalize=True)

pass_by_success_breakdown = pd.crosstab(
    dz_passes["detail"],
    dz_passes["outcome"],
    normalize="index"
)

print(f"The breakdown of defensive pass type frequencies overall is as follows: {pass_overall_breakdown}. ")
print(f"\nThe breakdown of defensive pass type frequencies stratified by pass success is as follows: {pass_by_success_breakdown}.")

In [ ]:
# Merge tracking for all players involved in these events
dz_tracking = dz_passes[["game_id", "sl_event_id", "player_id"]].merge(
    tracking,
    on=["game_id", "sl_event_id"],
    how="left"
)

dz_tracking.head()

# Identify passer rows
dz_tracking["is_passer"] = dz_tracking["player_id_x"] == dz_tracking["player_id_y"]

In [ ]:
### Compute distance to passer

# Extract passer coordinates per event
passer_coords = (
    dz_tracking[dz_tracking["is_passer"]][["game_id", "sl_event_id", "tracking_x", "tracking_y"]]
    .rename(columns={"tracking_x": "passer_x", "tracking_y": "passer_y"})
)

# Merge back to compute distance from passer for all players
dz_tracking = dz_tracking.merge(
    passer_coords,
    on=["game_id", "sl_event_id"],
    how="left"
)

# Compute distance to passer 
# TODO confirm computation
dz_tracking["dist_to_passer"] = np.sqrt(
    (dz_tracking["tracking_x"] - dz_tracking["passer_x"])**2 +
    (dz_tracking["tracking_y"] - dz_tracking["passer_y"])**2
)





In [ ]:
# Visualize distance to passer values
dz_tracking["dist_to_passer"].hist(bins=30)

In [ ]:
# Identify Opponents

dz_tracking.columns

In [ ]:
### Identify opponents 

#  Build event to passer_team_id mapping
passer_lookup = (
    dz_tracking.loc[dz_tracking["is_passer"], 
                    ["game_id", "sl_event_id", "team_id"]]
    .drop_duplicates(subset=["game_id", "sl_event_id"])
    .set_index(["game_id", "sl_event_id"])["team_id"]
)

# Map passer team back to all rows
dz_tracking["passer_team_id"] = list(
    zip(dz_tracking["game_id"], dz_tracking["sl_event_id"])
)
dz_tracking["passer_team_id"] = dz_tracking["passer_team_id"].map(passer_lookup)

# Identify opponents
dz_tracking["is_opponent"] = (
    dz_tracking["team_id"] != dz_tracking["passer_team_id"]
)

dz_tracking.columns



In [ ]:
dz_tracking.head()

In [ ]:

# Event-Level Pressure Summary
pressure_summary = (
    dz_tracking
    .query("is_opponent == True")
    .groupby(["game_id", "sl_event_id"])
    .agg(
        nearest_defender=("dist_to_passer", "min"),
        defenders_within_3m=("dist_to_passer", lambda x: (x < 3).sum())
    )
    .reset_index()
)

dz_passes = dz_passes.merge(
    pressure_summary,
    on=["game_id", "sl_event_id"],
    how="left"
)


In [ ]:
# Investigation I: Does Pressure Matter? - Exploration

sns.scatterplot(
    data=dz_passes,
    x="nearest_defender",
    y=(dz_passes["outcome"] == "successful").astype(int),
    alpha=0.1
)
plt.show()

In [ ]:
# Investigation I: Does Pressure Matter? - Exploration
dz_passes["success"] = (dz_passes["outcome"] == "successful").astype(int)

dz_passes["pressure_bin"] = pd.cut(
    dz_passes["nearest_defender"],
    bins=[0,1,2,3,5,10,50]
)

pressure_effect = (
    dz_passes
    .groupby("pressure_bin")
    .success
    .mean()
    .reset_index()
)

pressure_effect

In [ ]:
# Test system hypothesis:

team_context = (
    dz_passes
    .groupby("team_id")
    .agg(
        avg_nearest_defender=("nearest_defender", "mean"),
        pct_high_pressure=("nearest_defender", lambda x: (x < 2).mean()),
        raw_success=("success", "mean")
    )
    .sort_values("avg_nearest_defender")
)

team_context.head(teams_tot)

In [ ]:
# Quantify multicollinearity between "nearest_defender" and "defenders_within_3m"

dz_passes[["nearest_defender", "defenders_within_3m"]].corr()

In [ ]:
# Explore success prob across pass types and pressure bands

pd.crosstab(
    dz_passes["pressure_bin"],
    dz_passes["detail"],
    values=dz_passes["success"],
    aggfunc="mean"
).fillna(0)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Features and target
features = ["nearest_defender", "defenders_within_3m", "detail", "team_id",  "period_time"]
target = "success"

# Drop rows with missing target or features
dz_model_data = dz_passes[features + [target]].dropna()

X = dz_model_data[features]
y = dz_model_data[target]

# Identify categorical features
categorical_features = ["detail", "team_id"]
numeric_features = ["nearest_defender", "defenders_within_3m", "period_time"]

# Fix for player IDs that are not included in both train and test sets
OneHotEncoder(drop="first", handle_unknown="ignore")

# Column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)

# Logistic regression pipeline
pipeline = Pipeline([
    ("preproc", preprocessor),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit model
pipeline.fit(X_train, y_train)

# Predictions and probabilities
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

# Evaluation
print("Classification Report:")
print(classification_report(y_test, y_pred))

auc = roc_auc_score(y_test, y_prob)
print(f"ROC AUC: {auc:.3f}")

# Optional: create dataframe of predicted probabilities
dz_model_data_test = X_test.copy()
dz_model_data_test["actual_success"] = y_test
dz_model_data_test["predicted_prob"] = y_prob

dz_model_data_test.head()
